# Task 2.1 — Dataset Selection and Setup
**Paper:** One-Sided Support Vector Regression for Multiclass Cost-Sensitive Classification (Tu & Lin, ICML 2010)  
**Student:** Vaishnav Verma (230160)

## Dataset Justification

I use the **Wine dataset** from the UCI Machine Learning Repository (available via `sklearn.datasets.load_wine`). It contains 178 samples with 13 chemical analysis features across 3 cultivar classes. This dataset is appropriate for the OSSVR method because (1) it is a multiclass classification problem, matching the problem type OSSVR was designed for, (2) the original paper uses UCI datasets including Wine in its experimental evaluation, and (3) the 3-class structure allows me to define a meaningful non-uniform cost matrix where some misclassifications are more expensive than others (e.g., confusing a premium wine cultivar with a cheap one is more costly than the reverse). The main limitations compared to the paper's full evaluation are smaller scale (178 samples vs. thousands in some paper experiments) and fewer classes (3 vs. up to 10+ in some benchmarks), which means the cost structure is simpler and the method has fewer opportunities to demonstrate its advantage over baselines.

In [1]:
# === Configuration & Random Seeds ===
import numpy as np
import random

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print(f"Random seed set to: {RANDOM_SEED}")

Random seed set to: 42


In [2]:
# === Load Dataset ===
from sklearn.datasets import load_wine
import pandas as pd

wine = load_wine()
X = wine.data
y = wine.target

print(f"Dataset: Wine")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"Feature names: {wine.feature_names}")

Dataset: Wine
Number of samples: 178
Number of features: 13
Number of classes: 3
Class distribution: {np.int64(0): np.int64(59), np.int64(1): np.int64(71), np.int64(2): np.int64(48)}
Feature names: ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']


The Wine dataset has 178 samples, 13 features, and 3 classes — meeting the minimum requirements of 100 samples and 2 features specified in the exam.

In [3]:
# === Preprocessing: Standardisation ===
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Standardise features to zero mean, unit variance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Train class distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Test class distribution:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

Training set: 124 samples
Test set:     54 samples
Train class distribution: {np.int64(0): np.int64(41), np.int64(1): np.int64(50), np.int64(2): np.int64(33)}
Test class distribution:  {np.int64(0): np.int64(18), np.int64(1): np.int64(21), np.int64(2): np.int64(15)}


In [4]:
# === Cost Matrix Definition ===
# Non-uniform cost matrix for 3 classes.
# Interpretation: Class 0 = premium cultivar, Class 1 = mid-range, Class 2 = budget.
# Misclassifying premium (0) as budget (2) is very costly; reverse is less so.

K = len(np.unique(y))
cost_matrix = np.array([
    [0, 1, 5],   # True class 0: predicting 1 costs 1, predicting 2 costs 5
    [2, 0, 3],   # True class 1: predicting 0 costs 2, predicting 2 costs 3
    [10, 1, 0],  # True class 2: predicting 0 costs 10, predicting 1 costs 1
])

print("Cost Matrix C[y_true, y_pred]:")
print(cost_matrix)
print(f"\nDiagonal (correct predictions): {np.diag(cost_matrix)}")
print(f"Max cost: {cost_matrix.max()} (true=2, pred=0)")
print(f"Min non-zero cost: {cost_matrix[cost_matrix > 0].min()}")

Cost Matrix C[y_true, y_pred]:
[[ 0  1  5]
 [ 2  0  3]
 [10  1  0]]

Diagonal (correct predictions): [0 0 0]
Max cost: 10 (true=2, pred=0)
Min non-zero cost: 1


## Preprocessing Summary

1. **Feature standardisation:** All 13 features are scaled to zero mean and unit variance using `StandardScaler`. This is standard practice for kernel methods (SVM/SVR) to ensure all features contribute equally.
2. **Stratified train/test split:** 70% training (124 samples), 30% test (54 samples), preserving class proportions.
3. **Cost matrix:** A 3×3 non-uniform cost matrix is defined manually. The asymmetry ensures the problem is genuinely cost-sensitive — different errors carry different penalties, which is exactly the setting OSSVR is designed for.